# 02 — Single-pass Gemini CreativeIR baseline

One primary multimodal call: reference MP4 + normalized source metadata + versioned prompt -> CreativeIR v0.1.
No ffprobe facts, no shot detection, no transcript, no OCR, no multi-pass correction — a deliberate comparator for the multi-step pipeline.

Model ID, temperature and pricing come from config/environment (see `src/tiktok_analytics_factory/baseline/config.py`), never from this notebook.

In [ ]:
from pathlib import Path
import json

from tiktok_analytics_factory.baseline.config import load_baseline_config
from tiktok_analytics_factory.baseline.run import run_baseline, new_run_id

VIDEO_ID = "REFERENCE_VIDEO_ID"          # the reference video ingested in the ingestion notebook
RAW_DIR = Path(f"data/raw/{VIDEO_ID}")
VIDEO_PATH = RAW_DIR / "video.mp4"
METADATA_PATH = RAW_DIR / "metadata.json"

# Optional JSON config file; otherwise env vars BASELINE_MODEL_ID / GEMINI_API_KEY are used.
config_path = Path("baseline.local.json")
config = load_baseline_config(config_path if config_path.exists() else None)
print(json.dumps(config.to_dict(), indent=2))

## Reference video

In [ ]:
from IPython.display import Video

assert VIDEO_PATH.exists(), f"Reference video missing: {VIDEO_PATH}"
Video(str(VIDEO_PATH), width=360)

## Run (or load) the baseline

In [ ]:
# Set RUN_ID to an existing run to inspect it without spending a call;
# leave as None to execute exactly one new primary Gemini call.
RUN_ID = None

if RUN_ID is None:
    metadata = json.loads(METADATA_PATH.read_text())
    result = run_baseline(config, VIDEO_PATH, VIDEO_ID, metadata)
else:
    run_dir = Path(config.derived_root) / VIDEO_ID / "decompilation" / "single_pass" / RUN_ID
    result = {
        "run_id": RUN_ID,
        "directory": str(run_dir),
        "parsed": json.loads((run_dir / "response.parsed.json").read_text()),
        "validation": json.loads((run_dir / "validation.json").read_text()),
        **json.loads((run_dir / "usage.json").read_text()),
    }

result["validation"]

## Parsed CreativeIR

In [ ]:
print(json.dumps(result["parsed"], indent=2)[:8000])

## Usage / latency / cost

In [ ]:
usage = json.loads((Path(result["directory"]) / "usage.json").read_text())
usage

## Manual evaluation rubric (side-by-side)

Score all 11 categories with concrete evidence using `examples/evaluation/rubric_template_v0_1.json`,
against the hand-reviewed annotations in `examples/evaluation/reference_annotations_template_v0_1.json`.
The filled evaluation is persisted to `evaluation.json` in the run directory.

In [ ]:
from tiktok_analytics_factory.baseline.artifacts import RunArtifacts

rubric = json.loads(Path("examples/evaluation/rubric_template_v0_1.json").read_text())

# Fill in scores 1-5 + notes/evidence per category, then persist:
rubric["run_metadata"] = {
    "video_id": VIDEO_ID,
    "run_id": result["run_id"],
    "model_id": result.get("model_id") or config.model_id,
    "prompt_version": config.prompt_version,
    "evaluator": None,
    "evaluated_at": RunArtifacts.utc_now_iso(),
}
rubric["schema_validation_success"] = result["validation"]["valid"]
rubric["latency_seconds"] = usage.get("latency_seconds")
rubric["api_usage"] = usage.get("usage")
rubric["cost_usd"] = usage.get("cost_usd")

# Example:
# rubric["categories"]["visible_text_ocr"] = {
#     "score": 3,
#     "notes": "Missed the mid-video price overlay.",
#     "evidence": "Overlay visible at ~7.2s in reference annotations.",
# }

RunArtifacts(Path(result["directory"])).write_evaluation(rubric)
print("evaluation written to", Path(result["directory"]) / "evaluation.json")